# Potts model

In [7]:
import numpy as np

from collections import deque

## **Step 1:** Define Q states with values 1, 2, ..., Q.

In [8]:
Q_n = 10
Q = np.arange(1,Q_n+1)
delta_cronecker = np.diag([1 for i in range(Q_n)])

## **Step 2:** Initialize the starting configuration:
- Cold: All spins set to 1.
- Random: Each spin randomly assigned a value from 1 to Q with uniform distribution.

In [9]:
grid_size = 10

grid_cold = np.ones((grid_size, grid_size))
grid_rand = np.random.choice(Q, size=(grid_size,grid_size), replace=True)

print(grid_rand)

[[ 7  3  1 10  1  7 10  8  9  7]
 [ 9  2  1  1  3  1  3  7  2  5]
 [ 5  5 10  8  5  5  1  9  4  9]
 [ 4  8  2  8  5  5 10  3  2  4]
 [ 6  6 10  2  7  4  4  5  5  1]
 [ 3  7  8  2  4  3  3  9  2  5]
 [ 3  7  4  3  7  2  3  5  4  5]
 [ 3  2  4  5  3  9  7  6 10  5]
 [ 7 10 10  3 10  8  6  1  9  5]
 [ 7  4  1  9  9  8  9  6  2 10]]


## **Step 3**: Run the Metropolis and Wolff cluster algorithms for the Potts model, similar to the Ising model.

For spin flips, propose a new, different random value from 1 to Q.

Formulas to compute energy and magnetization:

+ Energy: $$ E= - \sum_{<i,j>} \delta_{s_i s_j}$$
where $<i,j>$ denotes neighboring pairs.

+ Magnetization:
$$ M = \dfrac{Q}{Q-1} \bigl( \dfrac{N_{max}}{N^2} - \dfrac{1}{Q}\bigr)$$
where $N$ is the lattice size, $N_{max}$  is the count of the most frequent spin state in the lattice.

### Func

In [45]:
def get_total_energy_system(grid):
    right_neighbours = np.roll(grid, 1, axis=1)
    down_neighbours = np.roll(grid, 1, axis=0)

    energy = -np.sum(grid == right_neighbours) - np.sum(grid == down_neighbours)

    return energy

get_total_energy_system(grid_rand)

-29

In [43]:
def get_total_magnetization_system(grid):
    _N = len(grid)
    _grid_counter = np.bincount(grid.flatten())[1:]
    _N_max = np.max(_grid_counter)

    return (Q_n / (Q_n-1)) * ((_N_max/(_N*_N)) - (1/Q_n))

get_total_magnetization_system(grid_rand)

0.055555555555555546

### MCMC

In [41]:
def local_spin_energy_change(grid, i, j):
    N = len(grid)
    old_spin = grid[i, j]
    _Q = [i for i in range(1,Q_n+1) if i != old_spin]

    new_spin = np.random.choice(_Q, size=1, replace=True).item()

    neighbours = (
        grid[(i-1)%N,j],
        grid[(i+1)%N,j],
        grid[i,(j-1)%N],
        grid[i,(j+1)%N]
    )

    old_energy = -sum(1 for n in neighbours if n == old_spin)
    new_energy = -sum(1 for n in neighbours if n == new_spin)

    delta_e = new_energy - old_energy

    return delta_e, new_spin

print(local_spin_energy_change(grid_rand, 3, 6), grid_rand[3,6])

(0, 2) 10


In [13]:
def monte_carlo_metropolis_step(grid, beta):
    def __boltzmann_probability(delta_e, beta):
        return np.exp(-beta * delta_e)

    N = len(grid)

    for _ in range(N * N):
        random_index = np.random.randint(N * N)
        x_ind = random_index // N
        y_ind = random_index % N

        delta_e, new_spin = local_spin_energy_change(grid, x_ind, y_ind)

        # Logika Metropolis
        if delta_e <= 0:
            grid[x_ind, y_ind] *= new_spin
        else:
            if np.random.random() <= __boltzmann_probability(delta_e, beta):
                grid[x_ind, y_ind] *= new_spin

    return grid

### Wolff

In [ ]:
def wolff_step(grid, beta, J=1.0):
    N = len(grid)
    random_index = np.random.randint(N * N)
    x_seed = random_index // N
    y_seed = random_index % N

    seed_spin = grid[x_seed, y_seed]
    cluster = {(x_seed, y_seed)}
    queue = deque([(x_seed, y_seed)])


    # accept prob
    acc_prob = 1.0 - np.exp(-2*beta*J)

    # neighbours checks
    while len(queue) > 0:
        x, y = queue.popleft()

        neighbours = (
            ((x-1)%N,y),
            ((x+1)%N,y),
            (x,(y-1)%N),
            (x,(y+1)%N)
        )

        # check neighbours
        for x_n, y_n in neighbours:
            # if in cluster or opposite spin -> we skip
            if ((x_n,y_n) not in cluster) and (grid[x_n,y_n] == seed_spin):
                if np.random.random() < acc_prob:
                    queue.append((x_n,y_n))
                    cluster.add((x_n,y_n))

    # cluster switching the spin
    _Q = [i for i in range(1,Q_n+1) if i != seed_spin]
    new_spin = np.random.choice(_Q, size=1, replace=True)
    for x, y in cluster:
        grid[x, y] = new_spin

    return grid

### Simulation loop

In [ ]:
def ising_simulation_with_stats(grid, step_func, beta, steps, burn_in_steps=0, verbose=False):
    active_steps = steps - burn_in_steps
    if active_steps <= 0:
        print("[WARNING] burn_in_steps is more or equal than the number of steps")
        return grid, 0, 0, 0

    _grid = grid.copy()
    _sum_mag = 0.0
    _sum_mag_sq = 0.0
    _sum_en = 0.0
    N_sq = len(grid)**2

    bar = range(steps)

    for i in bar:
        _grid = step_func(_grid, beta)

        if i >= burn_in_steps:
            # Obliczanie magnetyzacji
            current_m = get_total_magnetization_system(_grid)
            _sum_mag += current_m
            _sum_mag_sq += current_m**2

            current_e = get_total_energy_system(_grid)
            _sum_en += current_e

    # Statystyki końcowe
    _avg_mag = _sum_mag / active_steps
    _avg_mag_sq = _sum_mag_sq / active_steps
    _total_en = _sum_en / active_steps  # Średnia energia zamiast ostatniej wartości

    # Podatność magnetyczna: chi = beta * N^2 * (<m^2> - <m>^2)
    _susceptibility = beta * N_sq * (_avg_mag_sq - (_avg_mag**2))

    if verbose:
        print("Total magnetism (avg): ", _avg_mag,
              "Total energy (avg): ", _total_en,
              "Total susceptibility: ", _susceptibility, sep='\n')

    return _grid, _avg_mag, _total_en, _susceptibility

# Task
Plot magnetization and energy of Potts model as a function of $\beta$ near $\beta_c = \ln(1+ \sqrt{Q})$ using Metropolis and Wolff cluster.